### Title: 00g_text_matching
### Purpose: Text match NHANES ingredient descriptions in all NHANES participants to DFG2
### Date: August 27, 2024
### Author: Jules Larke

In [1]:
import pandas as pd
import string
import re
import nltk
nltk.data.path.append('/Users/local-jules/nltk_data')
wn = nltk.WordNetLemmatizer()

In [2]:
from polyfuzz.models import TFIDF
from polyfuzz import PolyFuzz

In [3]:
# fda_fndds_map_final.csv is a list of all unique ingredients in the all NHANES participants after mapping to the FDA-FDD. This was generated in the last cell in notebook 00a_generate_datasets.ipynb
nhanes = pd.read_csv('../../data/00/ingredient_matching/fda_fndds_map_final.csv', usecols=['Ingredient description', 'Ingredient code'])

# Load a list of all DFG2 foods (same foods in each of the other DFG2 datasets)
dfg = pd.read_csv('../../data/00/dfg2/monosaccharide_v6.csv', usecols=['sample_id', 'simple_name'])

# rename columns 
nhanes = nhanes.rename(columns={'Ingredient description':'ingred_desc', 'Ingredient code': 'ingred_code'})

In [4]:
punct = string.punctuation[0:11] + string.punctuation[13:] # remove '-' from the list of punctuation.
stopwords = ['','and', 'to', 'not', 'no',  'bkdfrd', 'ppd', 'pkgddeli', 'pkgd', 'xtra', 'oz', 'in', 'with', 'or', 'only', 'cooking', 'as', 'food', 'distribution', 'usda', 'form', 'w', 'wo', 'ns', 'nfs', 'incl']

def clean_text(text):
    text = "".join([word for word in text if word not in punct])
    tokens = re.split(r'[-\W+]', text)
    text = [word for word in tokens if word not in stopwords]
    text = [wn.lemmatize(word) for word in tokens if word not in stopwords]
    return "default" if text is [] else ' '.join(set(text))

nhanes['ingred_clean'] = nhanes['ingred_desc'].apply(lambda x: clean_text(x.lower()))
dfg['dfg_clean'] = dfg['simple_name'].apply(lambda x: clean_text(x.lower()))

In [5]:
# create lists of food descriptions for matching
nhanes_tokens = nhanes['ingred_clean'].to_list()
dfg_tokens = dfg['dfg_clean'].to_list()

In [6]:
# create the text matching model using TFIDF with tri-grams
tfidf = TFIDF(n_gram_range=(1, 3))
model = PolyFuzz(tfidf).match(nhanes_tokens, dfg_tokens)

In [7]:
# find matches between NHANES food descriptions and DFG2 food descriptions
match = model.get_matches()
match.rename(columns={'From':'ingred_clean', 'To':'dfg_clean'},inplace=True)

In [8]:
# merge the matches with the original descriptions for both NHANES and DFG2 (helps to visually see what the pre-cleaned description contained)
matched_1 = match.merge(nhanes, on='ingred_clean', how='left').drop_duplicates(subset='ingred_code')
matched_2 = matched_1.merge(dfg, on='dfg_clean', how='left').drop_duplicates(subset='ingred_code')

In [9]:
# select columns
matched_2 = matched_2[['ingred_clean', 'dfg_clean', 'ingred_desc', 'simple_name', 'ingred_code', 'sample_id', 'Similarity']]

In [10]:
matched_2.to_csv('../../data/00/ingredient_matching/nhanes_dfg2_map.csv', index=None)

#### nhanes_dfg2_map.csv will be manually checked and matches verified. The output file renamed to nhanes_dfg2_map_complete.csv

In [11]:
# load the ingredient descriptions matched to DFG2 descriptions
nhanes_matched = pd.read_csv('../../data/00/ingredient_matching/nhanes_dfg2_map_complete.csv', usecols=['ingred_desc', 'simple_name', 'ingred_code', 'match', 'replace'])

In [12]:
replace = nhanes_matched[nhanes_matched['replace'].notna()]
replace = replace[['ingred_code', 'ingred_desc', 'replace']]
replace.rename(columns={'replace':'simple_name'},inplace=True)

In [13]:
matched = nhanes_matched[nhanes_matched['match'].isna()]
matched = matched[['ingred_code', 'ingred_desc', 'simple_name']]

In [14]:
pd.concat([replace, matched]).to_csv('../../data/00/ingredient_matching/nhanes_dfg2_map_final.csv', index=None)